In [ ]:
!pip install beautifulsoup4 requests -q
print("✅ Done!")

In [ ]:
import requests
from bs4 import BeautifulSoup

# HBL FAQ page scrape karo
url = "https://www.hbl.com/personal/help-support"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

# Text extract karo
text = soup.get_text(separator="\n", strip=True)

# File mein save karo
with open("hbl_faq.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ HBL FAQ save ho gayi!")
print(f"Total characters: {len(text)}")
print("\n--- Preview (pehli 500 lines) ---")
print(text[:500])

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

urls = [
    "https://www.hbl.com/personal/help-support",
    "https://www.hbl.com/personal/digital-banking/hbl-mobile",
    "https://www.hbl.com/personal/digital-banking/hbl-digital-account",
    "https://www.hbl.com/personal/cards",
    "https://www.hbl.com/personal/accounts",
    "https://www.hbl.com/personal/loans",
    "https://www.hbl.com/about-us/contact-us",
]

all_text = ""

for url in urls:
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")

        # Sirf meaningful text lo
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        text = soup.get_text(separator="\n", strip=True)
        all_text += f"\n\n=== {url} ===\n\n{text}"
        print(f"✅ {url} — {len(text)} chars")
        time.sleep(1)
    except Exception as e:
        print(f"❌ Error: {url} — {e}")

with open("hbl_faq.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print(f"\n✅ Total data: {len(all_text)} characters")

In [ ]:
!pip install langchain langchain-community faiss-cpu sentence-transformers langchain-groq langchain-text-splitters streamlit pyngrok -q
print("✅ Done!")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("hbl_faq.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks ban gaye!")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("⏳ Embedding model load ho raha hai...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ Vector store ready!")

In [ ]:
app_code = '''
import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Page config
st.set_page_config(
    page_title="HBL Customer Support",
    page_icon="🏦",
    layout="centered"
)

# Custom CSS - Popup bubble style
st.markdown("""
<style>
    /* Main background */
    .stApp {
        background-color: #f0f2f5;
    }

    /* Chat container */
    .chat-container {
        max-width: 400px;
        margin: 50px auto;
        background: white;
        border-radius: 16px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.15);
        overflow: hidden;
    }

    /* Header */
    .chat-header {
        background: #00A651;
        color: white;
        padding: 16px 20px;
        display: flex;
        align-items: center;
        gap: 10px;
    }

    /* Messages */
    .user-msg {
        background: #00A651;
        color: white;
        padding: 10px 14px;
        border-radius: 18px 18px 4px 18px;
        margin: 8px 8px 8px auto;
        max-width: 75%;
        width: fit-content;
        float: right;
        clear: both;
    }

    .bot-msg {
        background: #f0f0f0;
        color: #333;
        padding: 10px 14px;
        border-radius: 18px 18px 18px 4px;
        margin: 8px auto 8px 8px;
        max-width: 75%;
        width: fit-content;
        float: left;
        clear: both;
    }

    /* Hide streamlit elements */
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

# Initialize
GROQ_KEY = "apni_groq_key_yahan_likho"

@st.cache_resource
def load_chain():
    loader = TextLoader("hbl_faq.txt", encoding="utf-8")
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )
    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 10}
    )

    prompt = PromptTemplate(
        template="""You are HBL Bank customer support assistant.
Answer clearly and helpfully using the context below.
If information is not available, say "Please call HBL at 111-111-425 for assistance."

Context: {context}
Question: {question}
Answer:""",
        input_variables=["context", "question"]
    )

    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        api_key=GROQ_KEY
    )

    def format_docs(docs):
        return "\\n\\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain

# Header
st.markdown("""
<div style="background:#00A651;padding:16px;border-radius:12px 12px 0 0;color:white;display:flex;align-items:center;gap:10px;">
    <span style="font-size:28px">🏦</span>
    <div>
        <b style="font-size:16px">HBL Support</b><br>
        <span style="font-size:12px;opacity:0.8">● Online | Typically replies instantly</span>
    </div>
</div>
""", unsafe_allow_html=True)

# Chat history
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "bot", "content": "Hi! 👋 Welcome to HBL Customer Support. How can I help you today?"}
    ]

# Display messages
for msg in st.session_state.messages:
    if msg["role"] == "bot":
        st.markdown(f"""
        <div style="background:#f0f0f0;padding:10px 14px;border-radius:18px 18px 18px 4px;
        margin:8px 60px 8px 8px;color:#333;font-size:14px;">
        🏦 {msg["content"]}
        </div>""", unsafe_allow_html=True)
    else:
        st.markdown(f"""
        <div style="background:#00A651;padding:10px 14px;border-radius:18px 18px 4px 18px;
        margin:8px 8px 8px 60px;color:white;font-size:14px;text-align:right;">
        {msg["content"]}
        </div>""", unsafe_allow_html=True)

# Quick buttons
st.markdown("<br>", unsafe_allow_html=True)
col1, col2, col3 = st.columns(3)
with col1:
    if st.button("💳 Cards"):
        st.session_state.quick = "Tell me about HBL credit cards"
with col2:
    if st.button("📱 Mobile App"):
        st.session_state.quick = "How to use HBL mobile banking?"
with col3:
    if st.button("🏧 ATM"):
        st.session_state.quick = "Where are HBL ATMs located?"

# Input
question = st.chat_input("Type your question...")

# Handle quick buttons
if "quick" in st.session_state:
    question = st.session_state.quick
    del st.session_state.quick

if question:
    st.session_state.messages.append({"role": "user", "content": question})

    with st.spinner("Finding answer..."):
        chain = load_chain()
        answer = chain.invoke(question)

    st.session_state.messages.append({"role": "bot", "content": answer})
    st.rerun()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py ready!")

In [ ]:
# app.py mein key update karo
with open("app.py", "r") as f:
    content = f.read()

from google.colab import userdata
groq_key = userdata.get("GROQ_API_KEY")

content = content.replace("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", groq_key)

with open("app.py", "w") as f:
    f.write(content)

print("✅ Key set ho gayi!")

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Ngrok auth token — free account banao: ngrok.com
ngrok.set_auth_token("3FPIMa6lEdBYFKRRSV5jDWHoO2t_7yxAKU8pjkKyCmwAMfKLn")

# App start karo
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
time.sleep(5)

# Public URL lo
tunnel = ngrok.connect(8501)
print(f"✅ App chal rahi hai!")
print(f"🌐 Link: {tunnel.public_url}")

In [ ]:
import time
time.sleep(10)  # 10 seconds wait karo
print("Ab browser refresh karo!")

In [ ]:
# app.py mein sahi key daalo
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")

with open("app.py", "r") as f:
    content = f.read()

# Key replace karo
content = content.replace("apni_groq_key_yahan_likho", groq_key)

with open("app.py", "w") as f:
    f.write(content)

print("✅ Key update ho gayi!")
print(f"Key starts with: {groq_key[:8]}...")

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Purana process band karo
subprocess.run(["pkill", "-f", "streamlit"])
time.sleep(3)

# Dobara start karo
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
time.sleep(8)

tunnels = ngrok.get_tunnels()
if tunnels:
    print(f"🌐 Link: {tunnels[0].public_url}")
else:
    tunnel = ngrok.connect(8501)
    print(f"🌐 Link: {tunnel.public_url}")

In [ ]:
import time
time.sleep(15)  # 15 seconds wait
print("✅ Ab browser mein refresh karo!")

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

urls = [
    "https://www.hbl.com/personal/help-support",
    "https://www.hbl.com/personal/digital-banking/hbl-mobile",
    "https://www.hbl.com/personal/digital-banking/hbl-digital-account",
    "https://www.hbl.com/personal/digital-banking/hbl-pay-billing-portal",
    "https://www.hbl.com/personal/cards",
    "https://www.hbl.com/personal/accounts",
    "https://www.hbl.com/personal/loans",
    "https://www.hbl.com/personal/raast",
    "https://www.hbl.com/personal/remittance",
    "https://www.hbl.com/about-us/contact-us",
    "https://www.hbl.com/personal/digital-banking/hbl-whatsapp-banking",
]

all_text = ""
for url in urls:
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        all_text += f"\n\n=== {url} ===\n\n{text}"
        print(f"✅ {url} — {len(text)} chars")
        time.sleep(1)
    except Exception as e:
        print(f"❌ {url}: {e}")

with open("hbl_faq.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print(f"\n✅ Total: {len(all_text)} characters")

In [ ]:
from google.colab import userdata
groq_key = userdata.get("GROQ_API_KEY")

app_code = '''import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time

st.set_page_config(
    page_title="HBL Customer Support",
    page_icon="🏦",
    layout="centered"
)

st.markdown("""
<style>
    .stApp { background-color: #f5f7fa; }
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}

    .chat-header {
        background: linear-gradient(135deg, #00A651, #007a3d);
        padding: 16px 20px;
        border-radius: 16px 16px 0 0;
        color: white;
        text-align: center;
    }

    .user-msg {
        background: #00A651;
        color: white;
        padding: 12px 16px;
        border-radius: 18px 18px 4px 18px;
        margin: 8px 8px 8px 60px;
        font-size: 14px;
        text-align: right;
    }

    .bot-msg {
        background: white;
        color: #333;
        padding: 12px 16px;
        border-radius: 18px 18px 18px 4px;
        margin: 8px 60px 8px 8px;
        font-size: 14px;
        box-shadow: 0 1px 4px rgba(0,0,0,0.1);
    }

    .stButton > button {
        background: white;
        color: #00A651;
        border: 1.5px solid #00A651;
        border-radius: 20px;
        padding: 6px 14px;
        font-size: 13px;
        width: 100%;
        transition: all 0.2s;
    }

    .stButton > button:hover {
        background: #00A651;
        color: white;
    }

    .status-dot {
        height: 10px;
        width: 10px;
        background-color: #90EE90;
        border-radius: 50%;
        display: inline-block;
        margin-right: 6px;
    }
</style>
""", unsafe_allow_html=True)

GROQ_KEY = "''' + groq_key + '''"

@st.cache_resource
def load_chain():
    loader = TextLoader("hbl_faq.txt", encoding="utf-8")
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 6, "fetch_k": 12}
    )
    prompt = PromptTemplate(
        template="""You are a professional HBL Bank customer support assistant.
Be helpful, polite and concise.
If user seems frustrated, acknowledge their concern first.
Use chat history to understand follow-up questions.
If information not available, say "Please call HBL at 111-111-425 for assistance."

Chat History:
{chat_history}

Context:
{context}

Question: {question}

Answer:""",
        input_variables=["context", "question", "chat_history"]
    )
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        api_key=GROQ_KEY
    )
    def format_docs(docs):
        return "\\n\\n".join(doc.page_content for doc in docs)
    return retriever, prompt, llm, format_docs

# Header
st.markdown("""
<div class="chat-header">
    <h3 style="margin:0;font-size:18px;">🏦 HBL Customer Support</h3>
    <p style="margin:4px 0 0 0;font-size:12px;opacity:0.9;">
    <span class="status-dot"></span>Online — Typically replies instantly</p>
</div>
""", unsafe_allow_html=True)

# Chat history
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "bot", "content": "Assalam o Alaikum! 👋 Welcome to HBL Customer Support. How can I help you today?"}
    ]

if "msg_count" not in st.session_state:
    st.session_state.msg_count = 0

# Messages display
for msg in st.session_state.messages:
    if msg["role"] == "bot":
        st.markdown(f\'<div class="bot-msg">🏦 {msg["content"]}</div>\', unsafe_allow_html=True)
    else:
        st.markdown(f\'<div class="user-msg">{msg["content"]}</div>\', unsafe_allow_html=True)

# Quick buttons
st.markdown("<br>", unsafe_allow_html=True)
col1, col2, col3 = st.columns(3)
with col1:
    if st.button("💳 Credit Cards"): st.session_state.quick = "Tell me about HBL credit cards"
with col2:
    if st.button("📱 Mobile App"): st.session_state.quick = "How to use HBL mobile banking?"
with col3:
    if st.button("🏧 ATM/Branch"): st.session_state.quick = "Where are HBL ATMs located?"

col4, col5, col6 = st.columns(3)
with col4:
    if st.button("💰 Loans"): st.session_state.quick = "Tell me about HBL loans"
with col5:
    if st.button("📤 Raast"): st.session_state.quick = "How to use Raast transfer?"
with col6:
    if st.button("🌍 Remittance"): st.session_state.quick = "How to receive money from abroad?"

# Stats
st.markdown("<br>", unsafe_allow_html=True)
st.markdown(f"""
<div style="background:white;padding:10px;border-radius:10px;text-align:center;
box-shadow:0 1px 4px rgba(0,0,0,0.1);font-size:12px;color:#666;">
💬 Messages: {st.session_state.msg_count} &nbsp;|&nbsp;
🤖 Powered by Llama 3.3 70B &nbsp;|&nbsp;
📞 HBL Helpline: 111-111-425
</div>
""", unsafe_allow_html=True)

# Input
question = st.chat_input("Type your message...")

if "quick" in st.session_state:
    question = st.session_state.quick
    del st.session_state.quick

if question:
    st.session_state.messages.append({"role": "user", "content": question})
    st.session_state.msg_count += 1

    chat_history = ""
    for msg in st.session_state.messages[-6:]:
        role = "User" if msg["role"] == "user" else "Assistant"
        chat_history += role + ": " + msg["content"] + "\\n"

    with st.spinner("HBL Support is typing..."):
        retriever, prompt, llm, format_docs = load_chain()
        docs = retriever.invoke(question)
        context = format_docs(docs)
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({
            "context": context,
            "question": question,
            "chat_history": chat_history
        })

    st.session_state.messages.append({"role": "bot", "content": answer})
    st.rerun()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ HBL App ready!")

In [ ]:
!pip install streamlit langchain langchain-community faiss-cpu sentence-transformers langchain-groq langchain-text-splitters beautifulsoup4 requests -q
print("✅ Done!")

In [ ]:
import subprocess, time
subprocess.run(["pkill", "-f", "streamlit"])
time.sleep(3)
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.headless=true"
])
time.sleep(8)
print("✅ Restart! Browser refresh karo!")